# 04 — Evaluate and decide

Week 4. The chosen model against the ladder, where it fails, the buffer the
app must show, and **one** look at July.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

from src.data import rebuild, split
from src.evaluate import (
    COVERAGE,
    bootstrap_ci,
    buffer_minutes,
    by_slice,
    coverage_at,
    distance_band,
    ladder,
    residuals,
    save_figure,
    score,
)
from src.features import build_features, feature_frame
from src.models import MedianDuration, TypicalSpeed, build_model, load


In [ ]:
df = build_features(rebuild())
parts = split(df)
train, val, test = parts["train"], parts["validation"], parts["test"]
y_train, y_val = train["duration_min"], val["duration_min"]
X_train, X_val = feature_frame(train), feature_frame(val)

chosen = load("chosen")
linreg = load("linear-regression")
val = val.assign(prediction=chosen.predict(X_val),
                 linear_prediction=linreg.predict(X_val))
print(chosen.named_steps["est"])

## The chosen model against the ladder, on June

In [ ]:
results = {
    "1. median of May": score(y_val, MedianDuration().fit(X_train, y_train).predict(X_val)),
    "2. distance / typical speed": score(
        y_val, TypicalSpeed().fit(train[["trip_distance"]], y_train).predict(val[["trip_distance"]])),
    "3. linear regression": score(y_val, val["linear_prediction"]),
    "4. chosen tree": score(y_val, val["prediction"]),
}
ladder(results).round(3)

## Where a straight line fails

The linear model assumes every extra mile costs the same number of minutes
everywhere, at every hour. The city does not work that way, and its residuals
say exactly where.

In [ ]:
val = val.assign(lin_resid=val["duration_min"] - val["linear_prediction"],
                 tree_resid=val["duration_min"] - val["prediction"],
                 pickup_hour_slice=val["tpep_pickup_datetime"].dt.hour,
                 distance_band=distance_band(val["trip_distance"]))

by_hour = val.groupby("pickup_hour_slice")[["lin_resid", "tree_resid"]].mean()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].axhline(0, lw=1, color="k")
axes[0].plot(by_hour.index, by_hour["lin_resid"], marker="o", label="linear")
axes[0].plot(by_hour.index, by_hour["tree_resid"], marker="o", label="tree")
axes[0].set(xlabel="hour of pickup", ylabel="mean (actual - predicted), min",
            title="A straight line is optimistic in the afternoon")
axes[0].legend()

by_band = val.groupby("distance_band", observed=True)[["lin_resid", "tree_resid"]].mean()
axes[1].axhline(0, lw=1, color="k")
axes[1].bar(np.arange(len(by_band)) - 0.2, by_band["lin_resid"], 0.4, label="linear")
axes[1].bar(np.arange(len(by_band)) + 0.2, by_band["tree_resid"], 0.4, label="tree")
axes[1].set(xticks=range(len(by_band)), xticklabels=by_band.index.astype(str),
            xlabel="distance band", title="and wrong at both ends of the distance range")
axes[1].legend()
save_figure(fig, "residuals-linear-vs-tree")
plt.show()

print(by_hour.round(2).to_string())
print()
print(by_band.round(2).to_string())

## The slices

Scores per group, **with the number of trips in each**. `needs_buffer` is
what that group would need on its own for 90% coverage.

In [ ]:
for col in ["pickup_hour_slice", "pickup_borough", "distance_band", "is_airport_trip"]:
    print(f"\n=== by {col} ===")
    print(by_slice(val, col).round(2).to_string(index=False))

In [ ]:
hour = by_slice(val, "pickup_hour_slice").sort_values("pickup_hour_slice")
band = by_slice(val, "distance_band")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].bar(hour["pickup_hour_slice"], hour["mae"])
axes[0].set(xlabel="hour of pickup", ylabel="MAE (min)", title="Error by hour, June")
axes[1].bar(band["distance_band"].astype(str), band["mae"])
axes[1].set(xlabel="distance band", title="Error by distance band, June")
save_figure(fig, "error-by-slice")
plt.show()

And the groups `02` promised to flag rather than drop — the flex fare
trips, the non-standard rate codes, and the negative amounts. None of them is
a feature; all of them are reported.

In [ ]:
for col in ["is_flex_fare", "is_non_standard_fare", "has_negative_amount"]:
    print(f"\n=== by {col} ===")
    print(by_slice(val, col, min_n=50).round(2).to_string(index=False))

## The decision — how large must the buffer be?

The app shows the predicted arrival time plus a buffer, so that 90% of
passengers arrive **before** the time shown. The buffer is the 90th
percentile of (actual − predicted) on June.

In [ ]:
buffer = buffer_minutes(val["duration_min"], val["prediction"], COVERAGE)
print(f"buffer for {COVERAGE:.0%} coverage, chosen on June: {buffer:.1f} minutes")
print(f"coverage with no buffer at all:                  "
      f"{coverage_at(val['duration_min'], val['prediction'], 0):.1%}")

print("\nwhat one global buffer actually delivers per group:")
for col in ["pickup_hour_slice", "pickup_borough", "distance_band"]:
    sl = by_slice(val, col, buffer=buffer)[[col, "n", "needs_buffer", "coverage_global"]]
    print(f"\n=== {col} ===")
    print(sl.sort_values("needs_buffer", ascending=False).round(2).to_string(index=False))

In [ ]:
# The ship threshold in PROBLEM.md is about the buffer too, not just the MAE.
rung1 = MedianDuration().fit(X_train, y_train)
rung2 = TypicalSpeed().fit(train[["trip_distance"]], y_train)

comparison = pd.DataFrame([
    {"model": "1. median of May",
     "june_mae": score(y_val, rung1.predict(X_val))["mae"],
     "buffer_90": buffer_minutes(y_val, rung1.predict(X_val))},
    {"model": "2. distance / typical speed",
     "june_mae": score(y_val, rung2.predict(val[["trip_distance"]]))["mae"],
     "buffer_90": buffer_minutes(y_val, rung2.predict(val[["trip_distance"]]))},
    {"model": "chosen tree",
     "june_mae": score(y_val, val["prediction"])["mae"],
     "buffer_90": buffer},
])
comparison["mae_gain_vs_rung2"] = comparison.loc[1, "june_mae"] - comparison["june_mae"]
comparison.round(2)

**Four groups fall below the 90% promise, and they are the same passengers
seen four ways.**

- **Queens: 83%.** The worst group by a distance, needing 10.8 minutes of
  buffer against the 6.7 it gets. Seventeen passengers in a hundred arrive
  after the time they were shown.
- **Airport trips: 81%**, needing 11.8 minutes. Queens is where two of the
  three airports are, so this is largely the same failure counted again.
- **Trips over 12 miles: 79%**, needing 13.1 — and 6–12 mile trips get 85%.
  Long trips are where the model's errors are largest in absolute terms.
- **The working day, 10:00 to 19:00: 87–89%**, worst from 15:00 to 17:00. The
  buffer is too small exactly when traffic is at its worst, and too generous
  at 05:00, where coverage reaches 97%.

The common thread is length: a long trip has more time in which to go wrong,
and Queens and the airports are long-trip groups.

**Our decision: the app should show a buffer per distance band, not one
global number.** Distance is known at the moment of prediction, the bands
already exist in `src.evaluate.distance_band`, and a per-band buffer fixes
the Queens and airport failures as a side effect rather than needing a rule
of its own. A single 6.7-minute buffer keeps the promise on average and
breaks it for the passengers with the most to lose — which is the kind of
average the model card is supposed to expose rather than hide.

The cost of that choice is honesty about the quote: a 12-mile trip would be
shown roughly 13 minutes of padding, which is a worse-looking estimate. We
would rather show a worse number that is true than a better one that is late
one time in five.

## July — the one look

Scored once, with a bootstrap interval next to the baseline. **Nothing is
tuned after this cell.** The buffer carried over is the one June chose, which
is the honest test of it.

In [ ]:
test = test.assign(prediction=chosen.predict(feature_frame(test)))
final = score(test["duration_min"], test["prediction"])
print("chosen model on July:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in final.items()})

baseline = MedianDuration().fit(X_train, y_train)
test = test.assign(baseline_prediction=baseline.predict(feature_frame(test)))

print()
for label, col in [("chosen model", "prediction"), ("median baseline", "baseline_prediction")]:
    for metric in ("mae", "rmse"):
        point, lo, hi = bootstrap_ci(test["duration_min"], test[col], metric=metric, n_boot=1000)
        print(f"{label:<16} {metric.upper():<5} {point:6.3f}   95% CI [{lo:.3f}, {hi:.3f}]")

In [ ]:
print(f"June's buffer, applied to July: {buffer:.1f} min")
print(f"coverage it achieved on July:   {coverage_at(test['duration_min'], test['prediction'], buffer):.1%}")
print(f"July would have needed:         "
      f"{buffer_minutes(test['duration_min'], test['prediction'], COVERAGE):.1f} min")

test = test.assign(distance_band=distance_band(test["trip_distance"]),
                   pickup_hour_slice=test["tpep_pickup_datetime"].dt.hour)
print("\nJuly by borough, with counts:")
print(by_slice(test, "pickup_borough", buffer=buffer).round(2).to_string(index=False))

## The decision

**Iterate — do not ship yet.**

The model clears the threshold in `PROBLEM.md` comfortably: 4.584 MAE on June
and **4.677 [4.654, 4.703] on July**, against the median baseline's 8.490
[8.449, 8.525], with intervals that do not overlap. July is close to June, so
nothing collapsed when the model met a month it had not seen. On the metric
alone, this works.

It should not ship as it stands for two reasons, and the first is the larger.

**The headline rests on a distance no app has.** `trip_distance` is measured
by the meter *during* the trip, so it already contains whatever detour
actually happened. Every number above is therefore an upper bound on what a
deployed version would do, and we do not know how far above. Until the model
is re-scored against a routing service's estimate of the planned route, the
honest statement is that we have measured a model we cannot yet deploy.

**The promise is not kept for everyone.** The buffer the app would show is
**6.7 minutes**, chosen on June, and it delivered 92.4% coverage on July —
but that average hides Queens at 83%, airport trips at 81%, and trips over
12 miles at 79%. A 90% promise that holds in Manhattan and breaks at the
airport is not a 90% promise.

**The one change most likely to help** is not a better model: it is replacing
`trip_distance` with a real route estimate and re-running this notebook. The
second is the per-band buffer argued for above. More tree depth is not on the
list — 14 and 16 were both worse on June.

What we would not do differently: the target definition, the cleaning rules,
or keeping the rate codes out of the training filter. Those are in
`PROBLEM.md`, and `MODEL_CARD.md` is written from this paragraph.